In [ ]:
import pandas as pd
import csv
import os

DATA_ROOT = "../../data"
full_data = pd.read_parquet(os.path.join(DATA_ROOT, "full_merged_data_filtered.parquet"))
full_data.head()

In [3]:
assert full_data.isna().mean().sum() == 0

In [4]:
# Languages of generated texts and original texts
# !pip install lingua-language-detector
from tqdm import tqdm
from lingua import Language, LanguageDetectorBuilder

languages = [
    Language.GERMAN,
    Language.ENGLISH,
    Language.SPANISH,  
    Language.GREEK,
    Language.POLISH,
    Language.PORTUGUESE,
    Language.CHINESE,
    Language.ARABIC,
    Language.RUSSIAN,
    Language.FRENCH,
    Language.PERSIAN,
    Language.UKRAINIAN,
    Language.ITALIAN,
    Language.BULGARIAN,
    Language.CATALAN,
    Language.KOREAN,
    Language.JAPANESE
]

detector = LanguageDetectorBuilder.from_languages(*languages).build()
UNKNOWN = "undefined"
def get_language(text):
    language = detector.detect_language_of(text)
    if language:
        return language.iso_code_639_1.name
    else:
        return UNKNOWN

generated_texts_languages, original_texts_languages = [], []
for i, row in tqdm(full_data.iterrows()):
    try:
        gen_lang = get_language(text=row['generated_text'])
        generated_texts_languages.append(gen_lang)
    except Exception as e:
        generated_texts_languages.append(UNKNOWN)

    try:
        orig_lang = get_language(text=row['real_text'])
        original_texts_languages.append(orig_lang)
    except Exception as e:
        original_texts_languages.append(UNKNOWN)

full_data['generated_text_language'] = generated_texts_languages
full_data['real_text_language'] = original_texts_languages

296580it [01:43, 2851.94it/s]


In [ ]:
from IPython.display import display

print("Number of languages in generated texts:", full_data['generated_text_language'].nunique())
print("Number of languages in real texts:", full_data['real_text_language'].nunique())
print("Number of samples where generated language is not the same as original:", 
      (full_data['generated_text_language'] != full_data['real_text_language']).sum(),
      round((full_data['generated_text_language'] != full_data['real_text_language']).mean() * 100) , "%")

print("Top 20 languages in generated texts:")
display(full_data['generated_text_language'].value_counts().head(15).to_frame('count').style.set_caption("Generated Text Languages (Top 15)"))

print("Top 20 languages in real texts:")
display(full_data['real_text_language'].value_counts().head(15).to_frame('count').style.set_caption("Real Text Languages (Top 15)"))

In [ ]:
# Language match per model and pipeline: 
full_data['language_match'] = full_data['generated_text_language'] == full_data['real_text_language']
language_match_stats = full_data.groupby(['model', 'pipeline'])['language_match'].mean().reset_index()
language_match_stats = language_match_stats.pivot(index='model', columns='pipeline', values='language_match').fillna(0)
print("Language Match Statistics by Model and Pipeline:")
display(language_match_stats)

In [ ]:
full_data[full_data['real_text_language']==UNKNOWN].real_text.value_counts().head(10)

In [ ]:
# Pairs of languages, that are comonly different: 
lang_pairs = full_data[full_data['generated_text_language'] != full_data['real_text_language']][['generated_text_language', 'real_text_language']]
# make tuples and count
lang_pairs_tuples = [tuple(x) for x in lang_pairs.to_numpy()]
lang_pairs_counts = pd.Series(lang_pairs_tuples).value_counts()
lang_pairs_counts[:10]

In [ ]:
# Rate of unknown languages in real texts:
unknown_rate = (full_data['real_text_language'] == UNKNOWN).mean()
print(f"Rate of unknown languages in real texts: {unknown_rate:.2%}")

# Rate of different languages between generated and real texts:
different_lang_rate = (full_data['generated_text_language'] != full_data['real_text_language']).mean()
print(f"Rate of different languages between generated and real texts: {different_lang_rate:.2%}")

# Rate of either real or generated is unknown or generated and real are different: 
either_unknown_or_different_rate = ((full_data['real_text_language'] == UNKNOWN) | (full_data['generated_text_language'] == UNKNOWN) | (full_data['generated_text_language'] != full_data['real_text_language'])).mean()
print(f"Rate of either real or generated is unknown or generated and real are different: {either_unknown_or_different_rate:.2%}")

In [ ]:
full_data["real_text_lenght"] = full_data["real_text"].apply(lambda x: len(x))
full_data["generated_text_lenght"] = full_data["generated_text"].apply(lambda x: len(x))

# Display lenghth statistics per pipeline and model:
length_stats = full_data.groupby(['model', 'pipeline'])[['generated_text_lenght']].agg(['mean'])
length_stats

# Filtering the data: 

In [ ]:
# Filter out samples where either real or generated is unknown or generated and real are different
full_data = full_data[~((full_data['real_text_language'] == UNKNOWN) | (full_data['generated_text_language'] == UNKNOWN) | (full_data['generated_text_language'] != full_data['real_text_language']))].reset_index(drop=True)
print(f"Cleaned data size: {full_data.shape[0]} samples")

In [ ]:
# Filtering messages with artifacts like "message_id =", "thread_id =", etc.
artifact_indicators = [x.lower() for x in ["message_id =", "thread_id =", "user_id =", "channel_id =", "[USER PERSONA]", "[CURRENT CONTEXT]", "USER_ID=", "[ORIGINAL POST]", "[THREAD]"]]
def contains_artifact(text):
    text_lower = text.lower()
    return any(indicator in text_lower for indicator in artifact_indicators)

# Rate of messages with artifacts
artifact_rate = full_data['generated_text'].apply(contains_artifact).mean()
print(f"Rate of messages with artifacts: {artifact_rate:.2%}")

# Filter out messages with artifacts
full_data = full_data[~full_data['generated_text'].apply(contains_artifact)].reset_index(drop=True)
print(f"Data size after removing artifacts: {full_data.shape[0]} samples")

In [ ]:
# Model Distribution
full_data.model.value_counts()

In [ ]:
len(full_data)

In [ ]:
# Number of Users, Threads, Channels
num_users = full_data["user_id"].nunique()
num_threads = full_data["thread_id"].nunique()
num_channels = full_data["channel"].nunique()

print(f"Number of unique users: {num_users}")
print(f"Number of unique threads: {num_threads}")
print(f"Number of unique channels: {num_channels}")

# Include mean, min, max
print("Average number of messages per user, model, pipeline:", full_data.groupby(['user_id', 'model', 'pipeline']).size().mean().round(1))
print("Minimum number of messages from a user, model, pipeline:", full_data.groupby(['user_id', 'model', 'pipeline']).size().min())
print("Maximum number of messages from a user, model, pipeline:", full_data.groupby(['user_id', 'model', 'pipeline']).size().max())

In [ ]:
# Length of generated texts and original texts

full_data['generated_text_length'] = full_data['generated_text'].apply(len)
full_data['real_text_length'] = full_data['real_text'].apply(len)

print("Average length of generated texts:", full_data['generated_text_length'].mean())
print("Average length of real texts:", full_data['real_text_length'].mean())


In [ ]:
# Presence of Links, Mentions, and Hashtags
def is_link(text):
    return "http://" in text or "https://" in text

def is_mention(text):
    return "@" in text

def is_hashtag(text):
    return "#" in text

full_data['generated_text_has_link'] = full_data['generated_text'].apply(is_link)
full_data['generated_text_has_mention'] = full_data['generated_text'].apply(is_mention)
full_data['generated_text_has_hashtag'] = full_data['generated_text'].apply(is_hashtag)
full_data['real_text_has_link'] = full_data['real_text'].apply(is_link)
full_data['real_text_has_mention'] = full_data['real_text'].apply(is_mention)
full_data['real_text_has_hashtag'] = full_data['real_text'].apply(is_hashtag)

print("Presence of Links, Mentions, and Hashtags in Generated Texts:")
print("Rate of generated texts vs. real texts containing links:", 
      round(full_data['generated_text_has_link'].mean() * 100, 2), "%", 
      round(full_data['real_text_has_link'].mean() * 100, 2), "%")
print("Rate of generated texts vs. real texts containing mentions:", 
      round(full_data['generated_text_has_mention'].mean() * 100, 2), "%", 
      round(full_data['real_text_has_mention'].mean() * 100, 2), "%")
print("Rate of generated texts vs. real texts containing hashtags:", 
      round(full_data['generated_text_has_hashtag'].mean() * 100, 2), "%", 
      round(full_data['real_text_has_hashtag'].mean() * 100, 2), "%")

In [20]:
full_data.drop(columns=['generated_text_length', 'real_text_length', 'generated_text_has_link',
       'generated_text_has_mention', 'generated_text_has_hashtag',
       'real_text_has_link', 'real_text_has_mention', 'real_text_has_hashtag'], inplace=True)

# Splitting the data logic (train-test)

In [ ]:
full_data.head(2)

In [22]:
import numpy as np

TEST_RATE = 0.25
MIN_NUM_USERS_PER_CHANNEL = 50 # (20%)

# Create a list of users per channel:
users_per_channel = {}
for _, record in full_data.iterrows():
    channel = record["channel"]
    user_id = record["user_id"]
    if channel not in users_per_channel:
        users_per_channel[channel] = set()
    users_per_channel[channel].add(user_id)

# Select 50 random users from each channel:
test_users = set()
for channel, users in users_per_channel.items():
    np.random.seed(42)
    number_of_users_to_select = np.min([np.max([MIN_NUM_USERS_PER_CHANNEL, int(len(users) * TEST_RATE)]), len(users)])
    selected_users = np.random.choice(list(users), size=number_of_users_to_select, replace=False)
    test_users = test_users | set(selected_users)
print(f"Total number of test users selected: {len(test_users)}")

Total number of test users selected: 1772


In [25]:
train_data = full_data[~full_data['user_id'].isin(test_users)]
test_data = full_data[full_data['user_id'].isin(test_users)]

# 10% of users in training set should be labeled as val (for threshold tuning):
VAL_RATE = 0.05
# Create a list of users per channel:
users_per_channel = {}
for _, record in train_data.iterrows():
    channel = record["channel"]
    user_id = record["user_id"]
    if channel not in users_per_channel:
        users_per_channel[channel] = set()
    users_per_channel[channel].add(user_id)

# Select 10% random users from each channel:
val_users = set()
for channel, users in users_per_channel.items():
    np.random.seed(42)
    number_of_users_to_select = int(len(users) * VAL_RATE)
    selected_users = np.random.choice(list(users), size=number_of_users_to_select, replace=False)
    val_users = val_users | set(selected_users)
print(f"Total number of val users selected: {len(val_users)}")
train_data["is_val"] = train_data['user_id'].isin(val_users)


train_data.to_parquet(os.path.join(DATA_ROOT, "train_data.parquet"))
test_data.to_parquet(os.path.join(DATA_ROOT, "test_data.parquet"))


Total number of val users selected: 212


/tmp/ipykernel_680/4065298426.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data["is_val"] = train_data['user_id'].isin(val_users)


In [ ]:
import glob

train_data = pd.read_parquet("../../data/train_data.parquet")
SELF_REVEAL_DIR = os.path.join(DATA_ROOT, "self_reveal_samples")

LANG_TO_ISO = {
    "English": "EN", "Spanish": "ES", "Ukrainian": "UK", "Russian": "RU",
    "German": "DE", "French": "FR", "Portuguese": "PT", "Polish": "PL",
    "Italian": "IT", "Arabic": "AR", "Chinese": "ZH", "Korean": "KO",
}

# Prefer the merged file; fall back to individual model files
merged = os.path.join(SELF_REVEAL_DIR, "self_reveal_all_lean.jsonl")
lean_files = [merged] if os.path.exists(merged) else glob.glob(
    os.path.join(SELF_REVEAL_DIR, "self_reveal_*_lean.jsonl")
)

if not lean_files:
    print("No self-reveal files found.")
else:
    sr_frames = [pd.read_json(f, lines=True) for f in lean_files]
    sr_raw = pd.concat(sr_frames, ignore_index=True).drop_duplicates(subset="generated_text")
    print(f"Loaded {len(sr_raw)} self-reveal samples from {len(lean_files)} file(s)")
    print(sr_raw["pattern_type"].value_counts().to_string())

    sr = pd.DataFrame({
        "model":                   sr_raw["model"],
        "pipeline":                "self_reveal",
        "real_text":               pd.NA,          # no paired human text
        "generated_text":          sr_raw["generated_text"],
        "user_id":                 pd.NA,
        "thread_id":               pd.NA,
        "platform":                "synthetic",
        "channel":                 "self_reveal_" + sr_raw["pattern_type"],
        "persona":                 pd.NA,
        "thread":                  pd.NA,
        "most_similar_threads":    pd.NA,
        "generated_text_language": sr_raw["language"].map(LANG_TO_ISO),
        "real_text_language":      pd.NA,
        "language_match":          pd.NA,
        "real_text_lenght":        pd.NA,
        "generated_text_lenght":   sr_raw["generated_text"].str.len(),
        "is_val":                  False,
    })

    train_data = pd.concat([train_data, sr], ignore_index=True)
    train_data.to_parquet(os.path.join(DATA_ROOT, "train_data_extended.parquet"))
    print(f"\ntrain_data after append : {len(train_data)} rows")
    print(f"  of which self-reveal   : {(train_data['pipeline'] == 'self_reveal').sum()}")
    print(f"  of which is_val=False  : {(~train_data['is_val']).sum()}")

# Final data stats: 

In [1]:
import pandas as pd 

train_data = pd.read_parquet("../../data/train_data.parquet")
test_data = pd.read_parquet("../../data/test_data.parquet")

In [2]:
full_data = pd.concat([train_data, test_data])

In [3]:
full_data["real_text_len"] = full_data["real_text"].apply(len)
full_data["generated_text_len"] = full_data["generated_text"].apply(len)

In [4]:
full_data["real_text_len"].mean()

np.float64(156.3238199655531)

In [5]:
full_data["generated_text_len"].mean()

np.float64(136.0386237926508)

In [6]:
full_data.groupby("pipeline")[["real_text_len", "generated_text_len"]].mean()

,real_text_len,generated_text_len
pipeline,,
full,155.177903,129.516734
reduced,157.522937,142.863299


In [7]:
# Presence of Links, Mentions, and Hashtags

def is_link(text):
    return "http://" in text or "https://" in text

def is_mention(text):
    return "@" in text

def is_hashtag(text):
    return "#" in text

full_data['generated_text_has_link'] = full_data['generated_text'].apply(is_link)
full_data['generated_text_has_mention'] = full_data['generated_text'].apply(is_mention)
full_data['generated_text_has_hashtag'] = full_data['generated_text'].apply(is_hashtag)
full_data['real_text_has_link'] = full_data['real_text'].apply(is_link)
full_data['real_text_has_mention'] = full_data['real_text'].apply(is_mention)
full_data['real_text_has_hashtag'] = full_data['real_text'].apply(is_hashtag)

print("Presence of Links, Mentions, and Hashtags in Generated Texts:")
print("Rate of generated texts vs. real texts containing links:", 
      round(full_data['generated_text_has_link'].mean() * 100, 2), "%", 
      round(full_data['real_text_has_link'].mean() * 100, 2), "%")
print("Rate of generated texts vs. real texts containing mentions:", 
      round(full_data['generated_text_has_mention'].mean() * 100, 2), "%", 
      round(full_data['real_text_has_mention'].mean() * 100, 2), "%")
print("Rate of generated texts vs. real texts containing hashtags:", 
      round(full_data['generated_text_has_hashtag'].mean() * 100, 2), "%", 
      round(full_data['real_text_has_hashtag'].mean() * 100, 2), "%")

Presence of Links, Mentions, and Hashtags in Generated Texts:
Rate of generated texts vs. real texts containing links: 0.62 % 4.2 %
Rate of generated texts vs. real texts containing mentions: 0.29 % 0.46 %
Rate of generated texts vs. real texts containing hashtags: 0.64 % 0.29 %


In [19]:
full_data.groupby("pipeline")['generated_text_has_link'].mean()*100

pipeline
full       0.947421
reduced    0.275606
Name: generated_text_has_link, dtype: float64

In [9]:
full_data.groupby("pipeline")['generated_text_has_mention'].mean()

pipeline
full       0.004793
reduced    0.000870
Name: generated_text_has_mention, dtype: float64

In [10]:
full_data.groupby("pipeline")['generated_text_has_hashtag'].mean()

pipeline
full       0.002975
reduced    0.009992
Name: generated_text_has_hashtag, dtype: float64

In [13]:
full_data.drop_duplicates(["user_id", "thread_id"]).shape

(73521, 25)

In [15]:
full_data.shape

(263594, 25)

In [14]:
full_data.user_id.nunique()

6326

In [16]:
test_data.user_id.nunique()

1772

In [17]:
test_data.shape

(71455, 16)

In [18]:
test_data.thread_id.nunique()

14288